# Model Analysis: PNN

Diagnostic analysis of the trained PNN (Probabilistic Neural Network) model. Includes a $\delta_b$ sensitivity sweep for the spoofing gain computation.

In [ ]:
import os, sys, glob, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import joblib

sys.path.insert(0, os.path.abspath(".."))

from detection.data.loaders import create_sequences, load_processed
from detection.models.pnn import PNN
from detection.spoofing.gain import calculate_expected_cost

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
DATA_DIR = os.path.join("..", "data", "processed", "TOTF.PA-book")
RESULTS_DIR = os.path.join("..", "results")
SEQ_LENGTH = 25
BATCH_SIZE = 64

LOB_COLUMNS = [
    f"{side}-{typ}-{lvl}"
    for lvl in range(1, 11)
    for side, typ in [("bid","price"),("bid","volume"),("ask","price"),("ask","volume")]
]

FILES = sorted(glob.glob(os.path.join(DATA_DIR, "*.parquet")))
TEST_FILE = FILES[22]
print(f"Test file: {os.path.basename(TEST_FILE)}")

In [ ]:
# Load feature names and model
feat_path = os.path.join(RESULTS_DIR, "pnn_features.txt")
with open(feat_path) as f:
    feat_names = [line.strip() for line in f if line.strip()]
num_features = len(feat_names)
print(f"Features: {num_features}")

# Load test data
df_test, features_test = load_processed(TEST_FILE, "xltime", LOB_COLUMNS)
feat_df = features_test.copy()
for col in feat_names:
    if col not in feat_df.columns:
        feat_df[col] = 0.0
feat_df = feat_df[feat_names]

scaler_path = os.path.join(RESULTS_DIR, "pnn_scaler.pkl")
scaler = joblib.load(scaler_path)
scaled = scaler.transform(feat_df.values.astype(np.float32)).astype(np.float32)
sequences = create_sequences(scaled, SEQ_LENGTH)

# Load PNN model
input_dim = SEQ_LENGTH * num_features
model = PNN(input_dim=input_dim, hidden_dim=64).to(DEVICE)
weights_path = os.path.join(RESULTS_DIR, "pnn_weights.pth")
model.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
model.eval()
print("PNN model loaded.")

# Get PNN predictions (mu, sigma, alpha)
all_mu, all_sigma, all_alpha = [], [], []
with torch.no_grad():
    for start in range(0, len(sequences), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(sequences))
        x = torch.tensor(sequences[start:end, -1, :], dtype=torch.float32).to(DEVICE)
        mu, sigma, alpha = model(x)
        all_mu.append(mu.cpu().numpy())
        all_sigma.append(sigma.cpu().numpy())
        all_alpha.append(alpha.cpu().numpy())

mu_arr = np.concatenate(all_mu).flatten()
sigma_arr = np.concatenate(all_sigma).flatten()
alpha_arr = np.concatenate(all_alpha).flatten()
print(f"Predictions: {len(mu_arr)} events")

## $\delta_b$ Sensitivity Sweep

Sweep the spoof order distance $\delta_b$ over {0, 0.001, 0.005, 0.01, 0.05} and plot the resulting spoofing gain distribution to verify sensitivity.

In [ ]:
DELTA_B_VALUES = [0.0, 0.001, 0.005, 0.01, 0.05]
Q = 4500        # Spoof order size
q = 100         # Genuine order size
DELTA_A = 0.0   # Genuine order distance
FEES = {"maker": 0.0, "taker": 0.0008}

# Compute spread from raw LOB data
spread_arr = (df_test["ask-price-1"].values - df_test["bid-price-1"].values)[SEQ_LENGTH:SEQ_LENGTH + len(mu_arr)]
spread_arr = np.maximum(spread_arr, 1e-4)  # floor to avoid zero spread

gain_results = {}

for delta_b in DELTA_B_VALUES:
    # Cost with spoof
    cost_with = calculate_expected_cost(
        mu_arr, sigma_arr, alpha_arr, spread_arr,
        delta_a=DELTA_A, delta_b=delta_b, Q=Q, q=q,
        epsilon_plus=FEES["maker"], epsilon_minus=FEES["taker"],
        side='ask'
    )
    # Cost without spoof (delta_b = 0 as baseline? Actually we compare vs no spoof)
    # The spoofing gain = cost_without - cost_with (reduction in cost)
    # But for the sweep, we compare the cost itself at different delta_b
    gain_results[delta_b] = cost_with

# Plot distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of costs at each delta_b
ax = axes[0]
for delta_b, costs in gain_results.items():
    costs_clipped = np.clip(costs, np.percentile(costs, 1), np.percentile(costs, 99))
    ax.hist(costs_clipped, bins=80, alpha=0.5, density=True, label=f"$\delta_b$={delta_b}")
ax.set_xlabel("Expected Cost")
ax.set_ylabel("Density")
ax.set_title("Spoofing Cost Distribution by $\delta_b$")
ax.legend(fontsize=8)

# Summary statistics
ax = axes[1]
delta_b_vals = sorted(gain_results.keys())
medians = [np.median(gain_results[d]) for d in delta_b_vals]
means = [np.mean(gain_results[d]) for d in delta_b_vals]
pct_positive = [100 * np.mean(gain_results[d] > 0) for d in delta_b_vals]
ax.plot(delta_b_vals, medians, 'o-', label='Median cost')
ax.plot(delta_b_vals, means, 's--', label='Mean cost')
ax.set_xlabel("$\delta_b$ (spoof distance)")
ax.set_ylabel("Expected Cost")
ax.set_title("Spoofing Cost Sensitivity to $\delta_b$")
ax.legend()

plt.tight_layout()
plt.show()

# Summary table
summary_rows = []
for d in delta_b_vals:
    c = gain_results[d]
    summary_rows.append({
        "$\delta_b$": d,
        "Mean": round(float(np.mean(c)), 6),
        "Median": round(float(np.median(c)), 6),
        "Std": round(float(np.std(c)), 6),
        "% positive": round(float(100 * np.mean(c > 0)), 2),
        "P99": round(float(np.percentile(c, 99)), 6),
    })
display(pd.DataFrame(summary_rows))